In [1]:
import sys
print(sys.version)

3.12.11 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 12:58:53) [MSC v.1929 64 bit (AMD64)]


In [2]:
!pip install langchain langchain-community langchain-openai langgraph
!pip install chromadb pypdf sentence-transformers
!pip install fastapi uvicorn python-multipart
!pip install pydantic python-dotenv
!pip install tiktoken openai

  Using cached tenacity-9.1.2-py3-none-any.whl.metadata (1.2 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached orjson-3.11.3-cp312-cp312-win_amd64.whl.metadata (43 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.33.2-cp312-cp312-win_amd64.whl.metadata (6.9 kB)
  Using cached aiohttp-3.12.15-cp312-cp312-win_amd64.whl.metadata (7.9 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.7.0-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached propcache-0.3.2-cp312-cp312-win_amd64.whl.metadata (12 kB)
  Using cached yarl-1.20.1-cp312-cp312-win_amd64.whl.metadata (76 kB)
  Using cached python_dotenv-1.1.1-py3-none-any.whl.metadata (24 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
 

In [3]:
import os
import json
from typing import List, Dict, Any, TypedDict, Annotated
from datetime import datetime
import operator

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import Document
from langchain_openai import ChatOpenAI

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

from fastapi import FastAPI, UploadFile, File, HTTPException
from pydantic import BaseModel, Field
import uvicorn

from pathlib import Path
import shutil
import re

C:\Users\Mohamed\miniconda3\envs\agentic\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [7]:
#@title Configuration

class Config:
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
    MODEL_NAME = "gpt-3.5-turbo"

    UPLOAD_DIR = Path("./data/uploads")
    VECTOR_DB_PATH = "./chroma_db"

    CHUNK_SIZE = 1000
    CHUNK_OVERLAP = 200
    EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

    MAX_QUESTIONS = 10
    TEMPERATURE = 0.7

    def __init(self):
        self.UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

In [8]:
config = Config()

In [9]:
#@title Pydantic Models

In [10]:
class QuestionRequest(BaseModel):
    query: str = Field(..., description="Topic or concept to generate questions about")
    num_questions: int = Field(default=5, ge=1, le=10)
    difficulty: str = Field(default="medium", pattern="^(easy|medium|hard)$")

In [11]:
class MCQQuestion(BaseModel):
    id: int
    question: str
    options: Dict[str, str]
    correct_answer: str
    explanation: str
    difficulty: str
    topic: str
    evaluation_score: float = 0.0

In [12]:
class QuestionResponse(BaseModel):
    questions: List[MCQQuestion]
    total_generated: int
    average_quality_score: float

In [13]:
class IngestResponse(BaseModel):
    status: str
    documnet_id: str
    table_of_contents: List[str]
    total_chunks: int

In [14]:
#@title PDF Processing Service

class PDFProcessor:
    def __init__(self, chunk_size: int=1000, chunk_overlap: int=200,
                 top_n_topics=20):
        self.__top_n_topics = top_n_topics
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", " ", ""]
        )

    def load_pdf(self, file_path: str) -> List[Document]:
        loader = PyPDFLoader(file_path)
        documnets = loader.load()

        chunks = self.text_splitter.split_documents(documnets)

        for i, chunk in enumerate(chunks):
            chunk.metadata.update({
                "chunk_id": i,
                "source": file_path,
                "processed_at": datetime.now().isoformat()
            })

        return chunks

    def extract_table_of_contents(self, documents: List[Document]) -> List[str]:

        toc = []

        for doc in documents:
            text = doc.page_content
            matches = re.findall(r'^\d+\.\s+([A-Z][^\n]+)', text, re.MULTILINE)
            toc.extend(matches)

        seen = set()
        toc_unique = []
        for item in toc:
            if item not in seen:
                seen.add(item)
                toc_unique.append(item)
        return toc_unique[:self.__top_n_topics]


In [15]:
pdf_processor = PDFProcessor(
    chunk_size=config.CHUNK_SIZE,
    chunk_overlap=config.CHUNK_OVERLAP
)

In [16]:
#@title Vector Store Service

class VectorStoreService:
    def __init__(self, persist_directory: str, embedding_model: str):
        self.persist_directory = persist_directory
        self.embeddings  = HuggingFaceEmbeddings(
            model_name=embedding_model,
            model_kwargs={"device": 'cpu'}
        )

    def create_vectorstore(self, documents: List[Document]) -> str:
        self.vectorstore = Chroma.from_documents(
            documents=documents,
            embedding=self.embeddings,
            persist_directory=self.persist_directory,
            collection_name="algebra_docs"
        )

        doc_id = f"doc_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        return doc_id

    def load_vectorstore(self):
        if not self.vectorstore:
            self.vectorstore = Chroma(
                persist_directory=self.persist_directory,
                embedding_function=self.embeddings,
                collection_name="algebra_docs"
            )

    def retrieve_context(self, query: str, k: int = 5) -> List[Document]:
        if not self.vectorstore:
            self.load_vectorstore()

        results = self.vectorstore.similarity_search(query, k=k)
        return results

In [17]:
vector_service = VectorStoreService(
    persist_directory=config.VECTOR_DB_PATH,
    embedding_model=config.EMBEDDING_MODEL
)

C:\Users\Mohamed\AppData\Local\Temp\ipykernel_18324\3042845684.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings  = HuggingFaceEmbeddings(
C:\Users\Mohamed\miniconda3\envs\agentic\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Mohamed\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_

In [18]:
#@title LangGraph State Definition

class AgentState(TypedDict):
    query: str
    num_questions: int
    difficulty: str
    retrieved_context: List[str]
    generated_questions: List[Dict[str, Any]]
    evaluated_questions: List[MCQQuestion]
    error: str

In [19]:
#@title Question Generator Agent

class QuestionGeneratorAgent:
    def __init__(self, llm):
        self.llm = llm
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an expert educational content creator specializing in mathematics.
Your task is to generate high-quality Multiple Choice Questions (MCQs) based on the provided context.

Guidelines:
- Create clear, unambiguous questions
- Ensure all options are plausible
- Only one option should be correct
- Include detailed explanations
- Match the specified difficulty level
- Base questions strictly on the provided context"""),
            ("user", """Context:
{context}

Generate {num_questions} MCQ questions about: {topic}
Difficulty level: {difficulty}

Return a JSON array with this exact structure:
[
  {{
    "question": "What is...",
    "options": {{
      "A": "Option 1",
      "B": "Option 2",
      "C": "Option 3",
      "D": "Option 4"
    }},
    "correct_answer": "B",
    "explanation": "Detailed explanation...",
    "topic": "Topic name"
  }}
]

IMPORTANT: Return ONLY the JSON array, no additional text.""")
        ])

    def generate(self, state: AgentState) -> AgentState:
        try:
            context = "\n\n".join(state["retrieved_context"])

            chain = self.prompt | self.llm
            response = chain.invoke({
                "context": context,
                "num_questions": state["num_questions"],
                "topic": state["query"],
                "difficulty": state["difficulty"]
            })

            content = response.content.strip()

            if content.startswith("```"):
                content = re.sub(r'^```json?\n|```$', '', content, flags=re.MULTILINE)

            questions = json.loads(content)

            for i, q in enumerate(questions):
                q["id"] = i + 1
                q["difficulty"] = state["difficulty"]

            state["generated_questions"] = questions

        except Exception as e:
            state["error"] = f"Question generation error: {str(e)}"
            state["generated_questions"] = []

        return state

In [20]:
class EvaluateAgent:
    def __init__(self, llm):
        self.llm = llm
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an expert educational content evaluator.
Evaluate the quality of MCQ questions based on these criteria:
1. Clarity (0-1): Is the question clear and unambiguous?
2. Correctness (0-1): Is the correct answer actually correct?
3. Distractor Quality (0-1): Are wrong options plausible?
4. Explanation Quality (0-1): Is the explanation clear and helpful?
5. Alignment (0-1): Does it match the difficulty level and topic?

Provide an overall score (0-1) and specific feedback."""),
            ("user", """Evaluate this question:

Question: {question}
Options: {options}
Correct Answer: {correct_answer}
Explanation: {explanation}
Difficulty: {difficulty}

Return JSON:
{{
  "overall_score": 0.0-1.0,
  "feedback": "Brief evaluation",
  "pass": true/false
}}

IMPORTANT: Return ONLY the JSON object.""")
        ])

    def evaluate(self, state: AgentState) -> AgentState:
        try:
            evaluated = []

            for question in state["generated_questions"]:
                chain = self.prompt | self.llm

                response = chain.invoke({
                    "question": question["question"],
                    "options": json.dumps(question["options"]),
                    "correct_answer": question["correct_answer"],
                    "explanation": question["explanation"],
                    "difficulty": question["difficulty"]
                })

                content = response.content.strip()
                if content.startswith("```"):
                    content = re.sub(r'^```json?\n|```$', '', content, flags=re.MULTILINE)

                evaluation = json.loads(content)

                if evaluation.get("pass", True) and evaluation["overall_score"] >= 0.6:
                    question["evaluation_score"] = evaluation["overall_score"]
                    evaluated.append(MCQQuestion(**question))

            state["evaluated_questions"] = evaluated

        except Exception as e:
            state["error"] = f"Evaluation error: {str(e)}"
            state["evaluated_questions"] = [
                MCQQuestion(**{**q, "evaluation_score": 0.7})
                for q in state["generated_questions"]
            ]

        return state


In [21]:
#@title LangGraph Workflow

class QuestionGenerationWorkflow:
    def __init__(self, vector_service: VectorStoreService, k: int=5):
        self.__k = k
        self.vector_service = vector_service
        self.llm = ChatOpenAI(
            model = config.MODEL_NAME,
            temperature=config.TEMPERATURE
        )

        self.question_agent = QuestionGeneratorAgent(self.llm)
        self.evaluator_agent = EvaluateAgent(self.llm)

        self.graph = self._build_graph()

    def _retrieve_node(self, state: AgentState) -> AgentState:
        try:
            docs = self.vector_service.retrieve_context(
                state["query"],
                k=self.__k,
            )
            state["retrieved_context"] = [doc.page_content for doc in docs]

        except Exception as e:
            state["error"] = f"Retrieval error: {str(e)}"
            state["retrieved_context"] = []

        return state

    def _build_graph(self) -> StateGraph:
        workflow = StateGraph(AgentState)

        workflow.add_node("retrieve", self._retrieve_node)
        workflow.add_node("generate", self.question_agent.generate)
        workflow.add_node("evaluate", self.evaluator_agent.evaluate)

        workflow.set_entry_point("retrieve")
        workflow.add_edge("retrieve", "generate")
        workflow.add_edge("generate", "evaluate")
        workflow.add_edge("evaluate", END)

        return workflow.compile()

    def run(self, query: str, num_questions: int=5, difficulty: str="medium") -> QuestionResponse:
        initial_state = AgentState(
            query=query,
            num_questions=num_questions,
            difficulty=difficulty,
            retrieved_context=[],
            generated_questions=[],
            evaluated_questions=[],
            error=""
        )

        final_state = self.graph.invoke(initial_state)

        if final_state["error"]:
            raise Exception(final_state["error"])

        questions = final_state["evaluated_questions"]
        avg_score = sum(q.evaluation_score for q in questions) / len(questions) if questions else 0

        return QuestionResponse(
            questions=questions,
            total_generated=len(questions),
            average_quality_score=avg_score
        )

In [22]:
app = FastAPI(
    title="RAG Question Generation API",
    description="Multi-agent system for generating MCQ questions from PDFs",
    version="1.0.0"
)

In [23]:
workflow = None

In [24]:
@app.on_event("startup")
async def startup_event():
    global workflow
    workflow = QuestionGenerationWorkflow(vector_service)
    print("API started successfully!")

C:\Users\Mohamed\AppData\Local\Temp\ipykernel_18324\981939747.py:1: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")


In [25]:
@app.post("ingest", response_model=IngestResponse)
async def ingest_pdf(file: UploadFile = File(...)):
    try:
        file_path = config.UPLOAD_DIR / file.filename
        with open(file_path, "wb") as buffer:
            shutil.copyfileobj(file.file, buffer)

        documents = pdf_processor.load_pdf(file_path)
        toc = pdf_processor.extract_table_of_contents(documents)

        doc_id = vector_service.create_vectorstore(documents)

        return IngestResponse(
            status="success",
            document_id=doc_id,
            table_of_contents=toc,
            total_chunks=len(documents)
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


In [26]:
@app.post("/generate/questions", response_model=QuestionResponse)
async def generate_questions(request: QuestionRequest):
    try:
        if not workflow:
            raise HTTPException(
                status_code=500,
                detail="Workflow not initialized"
            )

        response = workflow.run(
            query=request.query,
            num_questions=request.num_questions,
            difficulty=request.difficulty
        )

        return response

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


In [27]:
@app.get("/health")
async def health_check():
    return {"status": "healthy", "timestamp": datetime.now().isoformat()}


In [28]:
#@title Testing the System (Ingest PDF)

pdf_path = "../data/raw/A Quick Algebra Review (1).pdf"

if os.path.exists(pdf_path):
    documents = pdf_processor.load_pdf(pdf_path)
    toc = pdf_processor.extract_table_of_contents(documents)
    doc_id = vector_service.create_vectorstore(documents)

    print(f"PDF Ingested Successfully!")
    print(f"Document ID: {doc_id}")
    print(f"Total Chunks: {len(documents)}")
    print(f"\nTable of Contents:")
    for i, topic in enumerate(toc, 1):
        print(f"  {i}. {topic}")
else:
    print("PDF file not found. Please ensure 'A Quick Algebra Review (1).pdf' is in the current directory.")


PDF Ingested Successfully!
Document ID: doc_20251003_044055
Total Chunks: 54

Table of Contents:
  1. Simplifying Expressions 
  2. Solving Equations 
  3. Problem Solving 
  4. Inequalities 
  5. Absolute Values 
  6. Linear Equations 
  7. Systems of Equations 
  8. Laws of Exponents 
  9. Quadratics 
  10. Rationals 
  11. Radicals 
  12. Understand the problem 
  13. Define the variables 
  14. Write an equation 
  15. There is one very basic rule when dealing with roots.  If the root is even, you 


In [29]:
workflow = QuestionGenerationWorkflow(vector_service)

In [30]:
# Test queries
test_queries = [
    {"query": "quadratic equations", "num_questions": 3, "difficulty": "medium"},
    {"query": "solving linear equations", "num_questions": 3, "difficulty": "easy"},
    {"query": "laws of exponents", "num_questions": 2, "difficulty": "hard"}
]


In [31]:
all_generated_questions = []

In [32]:
for test_query in test_queries:
    print(f"\n\n Generating questions for: {test_query['query']}")
    print(f"   Difficulty: {test_query['difficulty']}, Count: {test_query['num_questions']}")
    print("-" * 60)

    try:
        response = workflow.run(**test_query)

        print(f"Generated {response.total_generated} questions")
        print(f"Average Quality Score: {response.average_quality_score}")

        for q in response.questions:
            all_generated_questions.append(q.dict())
            print(f"\nQuestion {q.id}: {q.question}")
            for key, value in q.options.items():
                marker = "✓" if key == q.correct_answer else " "
                print(f"   {marker} {key}. {value}")
            print(f"   Explanation: {q.explanation}")
            print(f"   Score: {q.evaluation_score:.2f}")

    except Exception as e:
        print(f"Error: {str(e)}")



 Generating questions for: quadratic equations
   Difficulty: medium, Count: 3
------------------------------------------------------------
Generated 3 questions
Average Quality Score: 0.7666666666666666

Question 1: What are the solutions to the quadratic equation 3x^2 - x = 0?
   ✓ A. x = -1, 0
     B. x = 0, 1/3
     C. x = -1/3, 0
     D. x = -1, 1/3
   Explanation: To solve the equation 3x^2 - x = 0, factor out x to get x(3x - 1) = 0. Set each factor to zero to find x = -1 and x = 0.
   Score: 0.80

Question 2: What are the roots of the quadratic equation x^2 - 5x + 6 = 0?
   ✓ A. x = 2, 3
     B. x = -2, -3
     C. x = 2, -3
     D. x = -2, 3
   Explanation: Factor the quadratic equation x^2 - 5x + 6 = 0 to get (x - 2)(x - 3) = 0. Set each factor to zero to find x = 2 and x = 3.
   Score: 0.80

Question 3: For the quadratic equation 2x^2 + 3x - 2 = 0, what are the solutions?
     A. x = -2, 1/2
   ✓ B. x = -1, 1/2
     C. x = -2, 1
     D. x = -1, 1
   Explanation: Factor the q

C:\Users\Mohamed\AppData\Local\Temp\ipykernel_18324\1964510438.py:13: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  all_generated_questions.append(q.dict())


Generated 1 questions
Average Quality Score: 0.75

Question 1: What is the solution to the linear equation 3x - 5 = 7?
   ✓ A. x = 2
     B. x = 4
     C. x = 6
     D. x = 12
   Explanation: To solve 3x - 5 = 7, we first add 5 to both sides: 3x = 12. Then, divide by 3 to find x = 4.
   Score: 0.75


 Generating questions for: laws of exponents
   Difficulty: hard, Count: 2
------------------------------------------------------------


C:\Users\Mohamed\AppData\Local\Temp\ipykernel_18324\1964510438.py:13: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  all_generated_questions.append(q.dict())


Generated 1 questions
Average Quality Score: 0.8

Question 2: Which of the following is equivalent to 5^-2?
   ✓ A. 1/25
     B. 25
     C. 1/5
     D. 5^2
   Explanation: A negative exponent indicates the reciprocal of the base raised to the positive exponent. Therefore, 5^-2 = 1/(5^2) = 1/25.
   Score: 0.80


C:\Users\Mohamed\AppData\Local\Temp\ipykernel_18324\1964510438.py:13: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  all_generated_questions.append(q.dict())


In [33]:
output_file = "sample_questions.json"
with open(output_file, "w") as f:
    json.dump(all_generated_questions, f, indent=2)


Starting FastAPI server...
API Documentation: http://localhost:8000/docs
Press Ctrl+C to stop


RuntimeError: asyncio.run() cannot be called from a running event loop

In [35]:
#@title Summary and Statistics

print("\n\n" + "=" * 60)
print("SYSTEM SUMMARY")
print("=" * 60)

summary = {
    "Total Documents Processed": len(documents) if 'documents' in locals() else 0,
    "Vector Store": "ChromaDB",
    "Embedding Model": config.EMBEDDING_MODEL,
    "LLM Model": config.MODEL_NAME,
    "Total Questions Generated": len(all_generated_questions),
    "Average Quality Score": round(sum(q["evaluation_score"] for q in all_generated_questions) / len(all_generated_questions), 2) if all_generated_questions else 0
}

for key, value in summary.items():
    print(f"{key}: {value}")

print("\nRAG Question Generation System Ready!")
print("=" * 60)



SYSTEM SUMMARY
Total Documents Processed: 54
Vector Store: ChromaDB
Embedding Model: sentence-transformers/all-MiniLM-L6-v2
LLM Model: gpt-3.5-turbo
Total Questions Generated: 5
Average Quality Score: 0.77

RAG Question Generation System Ready!
